In [19]:
import pandas as pd
import numpy as np

In [29]:
import numpy as np
import pandas as pd

# ---- Data Loader and Preprocessing ----

class Read_CSV:
    def __init__(self, path):
        self.path = path
        self.data = None
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.word_count = 2

    def read_csv(self):
        try:
            self.data = pd.read_csv(self.path)
            print("Data loaded successfully.")
        except FileNotFoundError:
            print('File path not found.')
        except Exception as e:
            print("Error:", e)

    def check_head(self, n=5):
        if self.data is not None:
            print(self.data.head(n))
        else:
            print("Data not loaded.")

    def clean_text(self, text):
        # Keep punctuation and line breaks; just strip carriage returns
        return text.replace('\r', '').strip()

    def fit_on_texts(self, texts):
        for text in texts:
            text = self.clean_text(text)
            for word in text.lower().split():
                if word not in self.word2idx:
                    self.word2idx[word] = self.word_count
                    self.idx2word[self.word_count] = word
                    self.word_count += 1

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            text = self.clean_text(text)
            seq = [self.word2idx.get(word, self.word2idx["<UNK>"]) for word in text.lower().split()]
            sequences.append(seq)
        return sequences

    def pad_sequences(self, sequences, maxlen=None, padding='pre', truncating='pre'):
        if maxlen is None:
            maxlen = max(len(seq) for seq in sequences)

        padded = []
        for seq in sequences:
            if len(seq) < maxlen:
                pad_len = maxlen - len(seq)
                pad = [self.word2idx["<PAD>"]] * pad_len
                seq = pad + seq if padding == 'pre' else seq + pad
            elif len(seq) > maxlen:
                seq = seq[-maxlen:] if truncating == 'pre' else seq[:maxlen]
            padded.append(seq)
        return np.array(padded)

    def vocab_size(self):
        return self.word_count


# ---- Simple LSTM Model from Scratch ----

class SimpleLSTM:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Initialize weights (using small random values)
        # Input gate weights
        self.W_i = np.random.randn(hidden_dim, input_dim) * 0.1
        self.U_i = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.b_i = np.zeros((hidden_dim, 1))
        
        # Forget gate weights
        self.W_f = np.random.randn(hidden_dim, input_dim) * 0.1
        self.U_f = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.b_f = np.zeros((hidden_dim, 1))
        
        # Output gate weights
        self.W_o = np.random.randn(hidden_dim, input_dim) * 0.1
        self.U_o = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.b_o = np.zeros((hidden_dim, 1))
        
        # Candidate memory cell weights
        self.W_c = np.random.randn(hidden_dim, input_dim) * 0.1
        self.U_c = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.b_c = np.zeros((hidden_dim, 1))
        
        # Output layer weights
        self.V = np.random.randn(output_dim, hidden_dim) * 0.1
        self.c = np.zeros((output_dim, 1))
        
        # To store activations for backward pass (not implemented here)
        self.last_h = None
        self.last_c = None

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))
    
    @staticmethod
    def dsigmoid(y):
        # derivative sigmoid given output y
        return y * (1 - y)
    
    @staticmethod
    def tanh(x):
        return np.tanh(x)
    
    @staticmethod
    def dtanh(y):
        return 1 - y**2
    
    def forward(self, inputs):
        """
        inputs: numpy array shape (seq_len, input_dim)
        returns: output logits for last time step, shape (output_dim, 1)
        """
        h_t = np.zeros((self.hidden_dim, 1))
        c_t = np.zeros((self.hidden_dim, 1))
        
        seq_len = inputs.shape[0]
        # Save intermediate states for backward (optional)
        self.last_h = []
        self.last_c = []
        
        for t in range(seq_len):
            x_t = inputs[t].reshape(-1,1)  # column vector
            
            i_t = self.sigmoid(self.W_i @ x_t + self.U_i @ h_t + self.b_i)
            f_t = self.sigmoid(self.W_f @ x_t + self.U_f @ h_t + self.b_f)
            o_t = self.sigmoid(self.W_o @ x_t + self.U_o @ h_t + self.b_o)
            c_hat_t = self.tanh(self.W_c @ x_t + self.U_c @ h_t + self.b_c)
            
            c_t = f_t * c_t + i_t * c_hat_t
            h_t = o_t * self.tanh(c_t)
            
            self.last_h.append(h_t)
            self.last_c.append(c_t)
        
        # Output layer (only last time step)
        y_t = self.V @ h_t + self.c  # logits
        
        # Return output logits and last hidden state (tuple)
        return y_t, h_t

    # NOTE: Backpropagation and training code would be needed here,
    # but omitted for brevity and complexity.
    # We will do a naive training loop with gradient approximation for demo.
    

# ---- Utility functions ----

def to_one_hot(y, vocab_size):
    one_hot = np.zeros((len(y), vocab_size))
    for i, val in enumerate(y):
        one_hot[i, val] = 1
    return one_hot

def tokens_to_one_hot_sequences(X, vocab_size):
    result = []
    for sequence in X:
        one_hot_seq = []
        for token_id in sequence:
            vec = np.zeros((vocab_size,))
            vec[token_id] = 1
            one_hot_seq.append(vec)
        result.append(np.array(one_hot_seq))
    return result  # list of (seq_len, vocab_size) arrays

def cross_entropy_loss(y_pred_logits, y_true_onehot):
    """
    y_pred_logits: shape (vocab_size, 1)
    y_true_onehot: shape (vocab_size,)
    """
    # Softmax
    exp_logits = np.exp(y_pred_logits - np.max(y_pred_logits))
    probs = exp_logits / np.sum(exp_logits)
    loss = -np.sum(y_true_onehot * np.log(probs + 1e-9))
    return loss, probs

def sample_with_temperature(logits, temperature=1.0):
    exp_logits = np.exp(logits / temperature)
    probs = exp_logits / np.sum(exp_logits)
    return np.random.choice(len(probs), p=probs)


# ---- Text Generation with Sampling ----

def generate_sequence_with_sampling(model, start_seq, idx2word, word2idx, vocab_size, length=50, temperature=1.0):
    current_seq = start_seq[:]
    generated_words = []
    
    for _ in range(length):
        one_hot_seq = []
        for token_id in current_seq:
            vec = np.zeros((vocab_size,))
            vec[token_id] = 1
            one_hot_seq.append(vec)
        one_hot_seq = np.array(one_hot_seq)
        
        y_pred_logits, _ = model.forward(one_hot_seq)
        logits = y_pred_logits.flatten()
        
        predicted_id = sample_with_temperature(logits, temperature)
        predicted_word = idx2word.get(predicted_id, "<UNK>")
        
        generated_words.append(predicted_word)
        
        current_seq = current_seq[1:] + [predicted_id]
        
        # Optionally stop if special token generated
        if predicted_word in ["<PAD>", "<UNK>"]:
            break
    return " ".join(generated_words)


# ---- Main script ----

def main():
    # Load and preprocess data
    csv = Read_CSV("/Users/bimalthapa/Downloads/5000_email_variations_dataset.csv")
    csv.read_csv()
    
    texts = csv.data['email_body']
    csv.fit_on_texts(texts)
    
    sequences = csv.texts_to_sequences(texts)
    maxlen = 50
    padded_sequences = csv.pad_sequences(sequences, maxlen=maxlen)
    
    print("Padded sequences shape:", padded_sequences.shape)
    
    # Prepare inputs and targets
    X = padded_sequences[:, :-1]  # all tokens except last
    y = padded_sequences[:, -1]   # last token
    
    vocab_size = csv.vocab_size()
    
    X_one_hot = tokens_to_one_hot_sequences(X, vocab_size)  # list of (seq_len, vocab_size)
    y_one_hot = to_one_hot(y, vocab_size)                   # (num_samples, vocab_size)
    
    # Initialize model
    model = SimpleLSTM(input_dim=vocab_size, hidden_dim=128, output_dim=vocab_size)
    
    # Simple training loop (VERY naive, one sample at a time, no backprop!)
    # This is a placeholder to show training structure.
    # Real training requires backprop and weight updates (not implemented here).
    epochs = 3
    learning_rate = 0.1
    
    for epoch in range(epochs):
        total_loss = 0
        for i in range(len(X_one_hot)):
            x_input = X_one_hot[i]          # shape (seq_len, vocab_size)
            y_true = y_one_hot[i].reshape(-1, 1)  # shape (vocab_size, 1)
            
            # Forward pass
            y_pred_logits, _ = model.forward(x_input)
            
            # Compute loss and probabilities
            loss, probs = cross_entropy_loss(y_pred_logits, y_true.flatten())
            total_loss += loss
            
            # NOTE: Backprop not implemented, so no weight updates
            
        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss / len(X_one_hot):.4f}")
    
    # Generate example text
    start_sequence = list(X[0])
    generated_text = generate_sequence_with_sampling(
        model,
        start_sequence,
        csv.idx2word,
        csv.word2idx,
        vocab_size,
        length=50,
        temperature=0.8
    )
    
    print("\nGenerated text:\n", generated_text)

if __name__ == "__main__":
    main()


Data loaded successfully.
Padded sequences shape: (5000, 50)
Epoch 1/3 Loss: 775.7897
Epoch 2/3 Loss: 775.7897
Epoch 3/3 Loss: 775.7897

Generated text:
 confident david, +81-3-1234-5678 brief or chat we’re requirements talent manager assist <PAD>


In [30]:
def postprocess_email_text(text):
    # Add line breaks after sentence-ending punctuation.
    import re
    # Replace dot, question, exclamation marks with punctuation + newline
    text = re.sub(r'([.?!])\s*', r'\1\n\n', text)
    
    # Capitalize first letter of each sentence
    sentences = [s.strip().capitalize() for s in text.split('\n') if s.strip()]
    
    # Join sentences with line breaks
    formatted_text = "\n".join(sentences)
    return formatted_text


In [35]:
def generate_sequence(model, start_seq, idx2word, word2idx, vocab_size, length=20):
    """
    Generate `length` words from model, starting from `start_seq` (list of token ids).
    
    Args:
        model: your SimpleLSTM model
        start_seq: list of ints, initial sequence (length = maxlen-1, e.g. 49 tokens)
        idx2word: dict mapping ids to words
        word2idx: dict mapping words to ids
        vocab_size: int, vocabulary size
        length: number of words to generate
    
    Returns:
        generated_words: list of generated words (strings)
    """
    current_seq = start_seq[:]  # copy to avoid modifying original
    generated_words = []
    
    for _ in range(length):
        # Convert current_seq to one-hot encoded input expected by your model
        one_hot_seq = []
        for token_id in current_seq:
            vec = np.zeros((vocab_size,))
            vec[token_id] = 1
            one_hot_seq.append(vec)
        one_hot_seq = np.array(one_hot_seq)  # shape (seq_len, vocab_size)
        
        # Get model prediction for next word
        y_pred = model.forward(one_hot_seq)
        output = y_pred[0].flatten()
        
        # Choose the predicted word ID with highest probability
        predicted_id = np.argmax(output)
        predicted_word = idx2word[predicted_id]
        
        generated_words.append(predicted_word)
        
        # Update current_seq: slide window by removing first token and adding predicted token
        current_seq = current_seq[1:] + [predicted_id]
    
    return generated_words


In [37]:
def add_email_structure(body_text):
    greeting = "Dear Team,\n\n"
    signoff = "\n\nBest regards,\nDavid"
    
    # Combine
    return greeting + body_text + signoff

raw_generated = generate_sequence_with_sampling(
    model,
    start_sequence,
    csv.idx2word,
    csv.word2idx,
    vocab_size,
    length=100,
    temperature=0.8
)

formatted = postprocess_email_text(raw_generated)
email_text = add_email_structure(formatted)

print("Email:\n", email_text)


NameError: name 'model' is not defined